In [ ]:
# On Google Colab, mount Drive and switch into the project directory.
# Locally this is a no-op (run the notebook from the `train/` directory).
try:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/Colab Notebooks')
except ModuleNotFoundError:
    pass

# Train LightGCN (BGE title-embedding initialization)

This notebook is the standard LightGCN pipeline, but the item embedding table (the layer-0 "ego" embeddings propagated through the user–item graph) is **warm-started from BGE title embeddings** instead of random init, as a cold-start mitigation. Everything else matches `train-lightgcn.ipynb`.

In [ ]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "scipy<1.12" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch" "sentence-transformers"

In [1]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# For SciPy 1.12+ compatibility: dok_matrix._update was removed
import scipy.sparse as sp
if not hasattr(sp.dok_matrix, '_update'):
    sp.dok_matrix._update = sp.dok_matrix.update

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [2]:
from typing import Any
import os
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
from sentence_transformers import SentenceTransformer

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty"
DATA_DIR: str = "../data"
SEED = 67

# if torch.cuda.is_available():
#     DEVICE = "cuda"
# elif torch.backends.mps.is_available():
#     DEVICE = "mps"
# else:
#     DEVICE = "cpu"

DEVICE = "cpu"

print(f"Using device: {DEVICE}")

Using device: cpu


## Create dataset

In [4]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "cold", "title"],
    },
    "embedding_size": 64,
    "epochs": 10,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "n_layers": 3,
    "reg_weight": 1e-4,
    "seed": SEED,
}

BGE_PATH: str = f"{DATA_DIR}/{DATASET_NAME}/bge_title_embeddings.pt"

config: Config = Config(model="LightGCN", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [6]:
dataset = create_dataset(config)


def _normalize_cold_feature(feat: pd.DataFrame, field: str = "cold") -> None:
    """Recover the literal 0/1 warm/cold labels for `feat[field]`.

    The `cold` column is declared as a TOKEN field, so RecBole remaps the
    binary labels onto a shared token vocabulary (e.g.
    ``{'[PAD]': 0, '0': 1, '1': 2}``) and applies that remap inconsistently
    across the user/item feats: one keeps the raw '0'/'1' strings while the
    other ends up with remapped ids. Both forms break here — the raw strings
    can't be cast to a LongTensor in ``data_preparation``, and the remapped
    ids no longer match the literal 0.0/1.0 the warm/cold eval compares
    against. Map every value back to its original label via the vocabulary.
    """
    token_of_id = {i: t for t, i in dataset.field2token_id[field].items()}

    def to_label(v: object) -> int:
        token = v if isinstance(v, str) else token_of_id.get(int(v), str(v))
        return 0 if token == "[PAD]" else int(token)

    feat[field] = feat[field].map(to_label).astype("int64")


_normalize_cold_feature(dataset.user_feat)
_normalize_cold_feature(dataset.item_feat)

train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
23 Jun 20:47    INFO  [Training]: train_batch_size = [1024] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
23 Jun 20:47    INFO  [Evaluation]: eval_batch_size = [409600000] 

## Precompute BGE title embeddings

In [ ]:
# Suppress httpx INFO logs from SentenceTransformer
logging.getLogger('httpx').setLevel(logging.WARNING)

logger = logging.getLogger()

if not os.path.exists(BGE_PATH):
    logger.info("Encoding %d item titles with BGE (BAAI/bge-base-en-v1.5)...", dataset.num("item_id"))

    title_tokens: torch.Tensor = dataset.item_feat["title"]
    id2token: dict[int, str] = {
        v: k for k, v in dataset.field2token_id["title"].items()
    }
    titles: list[str] = [id2token.get(tok.item(), "") for tok in title_tokens]

    bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)
    embs = bge_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
    embs = torch.from_numpy(embs).float()

    torch.save(embs, BGE_PATH)
    logger.info("BGE embeddings saved to %s", BGE_PATH)
else:
    logger.info("BGE embeddings already exist at %s, loading cached.", BGE_PATH)

23 Jun 20:47    INFO  Encoding 250853 item titles with BGE (BAAI/bge-base-en-v1.5)...
23 Jun 20:47    INFO  Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
Batches:  11%|█         | 103/980 [03:24<26:58,  1.85s/it]

## Initialize LightGCN with projected BGE embeddings and train

LightGCN reads `item_embedding.weight` as the layer-0 ego embeddings inside `get_ego_embeddings()`, so seeding it injects title content at the input of the graph convolution. We build the model normally (random init for the user side), then overwrite the item table before any forward pass — so the cached `restore_user_e`/`restore_item_e` propagated embeddings are still empty and nothing stale leaks in.

In [ ]:
bge_embs = torch.load(BGE_PATH, map_location="cpu")
bge_dim = bge_embs.shape[1]
emb_size = config["embedding_size"]

# Seeded random projection: BGE(768) -> embedding_size(64)
g = torch.Generator().manual_seed(SEED)
projection = torch.randn(bge_dim, emb_size, generator=g) / (bge_dim ** 0.5)

item_init_embs = bge_embs @ projection  # (n_items, emb_size)

# Zero out padding item (index 0) so it stays inert in graph propagation
item_init_embs[0] = 0.0

# Create standard LightGCN model (random init), then overwrite the
# item_embedding table with our BGE-derived vectors. The user_embedding is
# left at random init — consistent with item-only cold-start content.
model: LightGCN = LightGCN(config, train_data.dataset).to(config["device"])
with torch.no_grad():
    model.item_embedding.weight.data.copy_(item_init_embs.to(config["device"]))

trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

In [ ]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")

## Evaluate on test set

In [ ]:
test_result: dict[str, float] = trainer.evaluate(test_data, load_best_model=False)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

## Evaluate by warm/cold split

In [ ]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    rows.append({
        "Segment": label,
        "Interactions": int(mask.sum()),
        **results,
    })

rows = []

cold_to_label = {0.0: "warm", 1.0: "cold"}

def evaluate_by_column(entity_feat, id_field, inter_id_array, entity_name):
    id_to_cold = dict(zip(
        entity_feat[id_field].numpy(),
        entity_feat["cold"].numpy(),
    ))
    for cold_val, label in cold_to_label.items():
        ids = {eid for eid, c in id_to_cold.items() if c == cold_val}
        mask = np.isin(inter_id_array, list(ids))
        if not mask.any():
            print(f"  {entity_name}-{label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"{entity_name}-{label}")

# Evaluation by user segments
evaluate_by_column(
    dataset.user_feat, dataset.uid_field,
    test_data.dataset.inter_feat[dataset.uid_field].numpy(),
    "user"
)

# Evaluation by item segments
evaluate_by_column(
    dataset.item_feat, dataset.iid_field,
    test_data.dataset.inter_feat[dataset.iid_field].numpy(),
    "item"
)

# Cross-tabulation: user × item segments
uid_to_cold = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["cold"].numpy(),
))
iid_to_cold = dict(zip(
    dataset.item_feat[dataset.iid_field].numpy(),
    dataset.item_feat["cold"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()
iid_array = test_data.dataset.inter_feat[dataset.iid_field].numpy()

for uc_val, uc_label in cold_to_label.items():
    for ic_val, ic_label in cold_to_label.items():
        uc_uids = {uid for uid, c in uid_to_cold.items() if c == uc_val}
        ic_iids = {iid for iid, c in iid_to_cold.items() if c == ic_val}
        mask = np.isin(uid_array, list(uc_uids)) & np.isin(iid_array, list(ic_iids))
        if not mask.any():
            print(f"  user-{uc_label}×item-{ic_label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"user-{uc_label}×item-{ic_label}")

# Display results sorted by NDCG
df_results = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False)
display(df_results)